# Processing: player-game dataset

Turns the EDA's decisions (`../02_eda/eda_decisions.yml`) into model-ready data for LightGBM, using the bundled
`dsprep` library (`../dsprep`) configured by `processing_player_games.py`. Target: `act` (PPR points), regression.

In [1]:
import pandas as pd
import processing_player_games as pp

dp = pp.dp
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 90)
TARGET = "act"
OUT, MODELS = "../data/processed", "../models"

train_raw = pd.read_parquet("../data/player_games_train.parquet")
test_raw = pd.read_parquet("../data/player_games_test.parquet")
dec = dp.load_decisions("../02_eda/eda_decisions.yml")
print(train_raw.shape, test_raw.shape)

(31269, 75) (3069, 75)


## 1. Decisions
The EDA's recommendations are taken as the defaults. Two changes are recorded as overrides: `season` is not a
model feature, and the weekday part of `kickoff` is skipped. The EDA's open questions are answered with the
defaults the research uses.

In [2]:
summary = dp.decisions_summary(dec)
print(summary.groupby(["section", "status"]).size())
print("\nopen questions:")
for q in dp.open_questions(dec):
    print(" -", q)
dec = dp.apply_overrides(dec, {
    "recommendations.datetime_parts": [],
    "recommendations.target.transform": None,
})
NOT_FEATURES = ["season"]  # kept in the clean frames for splitting and evaluation only

section          status     
cleaning         applied         2
                 recommended     1
recommendations  drop            1
                 keep            6
                 recommended    19
dtype: int64

open questions:
 - Sleeper's final snapshot knows game-day inactives (it projects 0 for 96% of players who end up not playing). Should the headline comparison include those rows (fair to the use case) or only players who played (fair to the model)? Both are reported.


**Findings**


- 26 decisions (3 cleaning, 23 recommendations), all taken as is.
- **Overrides:** `season` is not a feature (no trend, and a tree can't extrapolate an unseen season), and the target stays untransformed.
- **Open question, answered with the research default:** Sleeper's snapshot knows game-day inactives. The model is scored against Sleeper both on all rows (the real use case) and on players who played (performance only), so both readings are available.

## 2. Cleaning
Re-applies the EDA's cleaning to the raw files. The test file was delivered already split, so `drop_rows=False`
keeps every test row.

In [3]:
train_all, clean_log = dp.apply_cleaning(train_raw, dec)
test, test_log = dp.apply_cleaning(test_raw, dec, drop_rows=False)
print(clean_log[["step", "column", "action", "n_affected"]].to_string(index=False) if "n_affected" in clean_log else clean_log.to_string(index=False))
print(train_all.shape, test.shape)

          step          column  action  n_affected
   null_tokens practice_status  to NaN        26.0
invalid_values   snap_share_l3 set NaN         4.0
       leakage         kickoff dropped         NaN
  drop_columns            name dropped         NaN
  drop_columns         gsis_id dropped         NaN
  drop_columns            team dropped         NaN
  drop_columns        opponent dropped         NaN
  drop_columns      rec_tgt_l3 dropped         NaN
  drop_columns      rec_tgt_l8 dropped         NaN
  drop_columns  qb_cpoe_rel_l8 dropped         NaN
(31269, 67) (3069, 67)


**Findings**


- Re-applied to both files: 26 whitespace practice tags become NaN, and 4 `snap_share_l3` values above 1 become NaN.
- 7 columns dropped: `kickoff`, `name`, `gsis_id`, `team`, `opponent`, `rec_tgt_l3`, `rec_tgt_l8`. No rows dropped. 45 columns remain in train and test.

## 3. Split
Out-of-time, by season, to match production (predict next weeks from past seasons): train 2022-2024,
validation 2025, test 2026 weeks 1-7 (the delivered file). History features were built before the split by
`build_dataset.py` and are past-only, so they are safe. Everything fitted here is fitted on train only.

In [4]:
splits = {"train": train_all[train_all.season <= 2024], "val": train_all[train_all.season == 2025], "test": test}
dp.split_report(splits, target=TARGET, group_col="player_id")

,split,rows,share,target_mean,groups_also_in_train
0,train,23480,0.683791,5.811939,NaN
1,val,7789,0.226833,5.764137,487.0
2,test,3069,0.089376,5.888413,383.0


**Findings**


- Train 23,480 rows (2022-2024), val 7,789 (2025), test 3,069 (2026 weeks 1-7). Target means are 5.81 / 5.76 / 5.89.
- 487 val players and 383 test players also appear in train, as expected for a new season of known players. The split is by time.

## 4. Target
No transform (see the EDA, section 4): the model predicts expected points directly.

In [5]:
tp = dp.TargetProcessor("regression", transform=None)
y = {"train": tp.fit_transform(splits["train"][TARGET])}
y |= {k: tp.transform(splits[k][TARGET]) for k in ("val", "test")}
print(tp.describe())

{'task': 'regression', 'mode': 'auto', 'transform': None}


## 5. Features
Spec reviewed from `dp.suggest_specs(dec, "gbm")`. Numeric columns are left as is (NaN kept, LightGBM handles it),
with missing indicators where the EDA found informative missingness (no history, unranked, undrafted). Categoricals
use native `category` dtype; rare-level grouping is off, because "Doubtful" is rare but almost never plays.
Two domain features, `injury_level` and `practice_level`: ordinal encodings of the injury report, added after
the first model could not isolate the rare "Doubtful" level. **No weekly projection is an input**, and the
check below fails loudly if one ever appears.

In [6]:
print(dp.to_python_literal(dp.suggest_specs(dec, model_family="gbm"))[:600], "...")
frames = {k: pp.add_domain_features(v) for k, v in splits.items()}  # ordinal injury features
fp = pp.build_feature_processor("gbm")
X = {"train": fp.fit_transform(frames["train"], y["train"])}
X |= {k: pp.align_categories(fp.transform(frames[k]), X["train"]) for k in ("val", "test")}
print({k: v.shape for k, v in X.items()})
cats = {c: [list(X[k][c].cat.categories) == list(X["train"][c].cat.categories) for k in X] for c in fp.cat_features()}
print("category levels identical across train/val/test:", cats)
print("concat keeps dtypes:", pd.concat(list(X.values())).dtypes.value_counts().to_dict())
print("unused input columns:", fp.unused_columns_)
assert not [c for c in X["train"] if "proj" in c or "sleeper_resid" in c], "weekly projection found in features"
fp.log

{'model_family': 'gbm',
 'fold_columns': {'group_col': None, 'time_col': None},
 'numeric': {'total_line': None,
             'team_spread': None,
             'implied_total': None,
             'age': None,
             'years_exp': None,
             'pts_l3': None,
             'snap_share_l3': None,
             'tgt_share_l3': None,
             'rush_att_l3': None,
             'pass_att_l3': None,
             'rz_opps_l3': None,
             'rec_air_yd_l3': None,
             'pts_l8': {'missing_indicator': True},
             'snap_share_l8': None,
             'tgt_share_l8': None, ...


{'train': (23480, 71), 'val': (7789, 71), 'test': (3069, 71)}
category levels identical across train/val/test: {'position': [True, True, True], 'roof': [True, True, True], 'report_status': [True, True, True], 'practice_status': [True, True, True]}
concat keeps dtypes: {dtype('float64'): 62, dtype('int8'): 5, CategoricalDtype(categories=['QB', 'RB', 'TE', 'WR'], ordered=False, categories_dtype=str): 1, CategoricalDtype(categories=['closed', 'dome', 'open', 'outdoors'], ordered=False, categories_dtype=str): 1, CategoricalDtype(categories=['Doubtful', 'Note', 'Questionable', '__MISSING__'], ordered=False, categories_dtype=str): 1, CategoricalDtype(categories=['Did Not Participate In Practice', 'Full Participation in Practice', 'Limited Participation in Practice', '__MISSING__'], ordered=False, categories_dtype=str): 1}
unused input columns: ['season', 'player_id', 'act']


,step,processor,column,action,detail
0,num,NumericProcessor,pts_l8,missing_indicator,train missing rate 3.867%
1,num,NumericProcessor,pts_prev_season,missing_indicator,train missing rate 21.222%
2,num,NumericProcessor,xfp_l8,missing_indicator,train missing rate 5.179%
3,num,NumericProcessor,fp_ecr,missing_indicator,train missing rate 17.521%
4,num,NumericProcessor,sleeper_adp,missing_indicator,train missing rate 29.940%
5,cat,CategoricalProcessor,position,native,4 levels kept as category
6,cat,CategoricalProcessor,roof,native,4 levels kept as category
7,cat,CategoricalProcessor,report_status,native,4 levels kept as category
8,cat,CategoricalProcessor,practice_status,native,4 levels kept as category


In [7]:
fdict = fp.feature_dictionary()
fdict.to_csv("feature_dictionary_gbm.csv", index=False)
fdict

,step,feature,source,processor,method,dtype,params
0,num,team_spread,team_spread,NumericProcessor,auto,float64,method=auto
1,num,total_line,total_line,NumericProcessor,auto,float64,method=auto
2,num,implied_total,implied_total,NumericProcessor,auto,float64,method=auto
3,num,home,home,NumericProcessor,auto,float64,method=auto
4,num,div_game,div_game,NumericProcessor,auto,float64,method=auto
...,...,...,...,...,...,...,...
66,num,practice_level,practice_level,NumericProcessor,auto,float64,method=auto
67,cat,position,position,CategoricalProcessor,native,category,method=native
68,cat,roof,roof,CategoricalProcessor,native,category,method=native
69,cat,report_status,report_status,CategoricalProcessor,native,category,method=native


**Findings**


- **71 features:** 62 numeric (NaN kept): history, ordinal injury levels, next man up (`team_pos_rank`, `team_pos_share`, `vacated_pts`, `is_expected_qb`), player efficiency (WOPR, EPA), opponent defense strength (league-relative and schedule-adjusted), own-team offense and the expected QB's passing ability (league-relative), 5 missing indicators (no history 3.9%, no last season 21.2%, no expected-points history 5.2%, **unranked 17.5%, undrafted 29.9%**), and 4 native categoricals.
- **No weekly projection is an input:** an assert fails the notebook if a `proj`/`sleeper_resid` column ever appears.
- Unused inputs are exactly the non-features: `season`, `player_id`, `act`.
- Category levels are identical across the splits (`align_categories`), and "Doubtful" keeps its own level.

## 6. Screening
Constant, duplicate and highly correlated features, and a single-feature leakage scan.

In [8]:
scr = dp.screening_report(X["train"], y["train"], "regression")
for k, v in scr.items():
    print(f"--- {k}: {len(v) if hasattr(v, '__len__') else v}")
    if hasattr(v, "head") and len(v):
        print(v.head(10).to_string())

--- constant: 0
--- duplicates: 0
--- correlated: 3
      feature_a        feature_b      corr
0   fp_pos_rank  sleeper_pos_adp  0.954845
1    pts_std_l8       pts_max_l8  0.954717
2  tgt_share_l8          wopr_l8  0.950277
--- leakage: 71
          feature     kind       measure   score  suspect
0  pts_season_avg  numeric  abs_spearman  0.6266    False
1          pts_l8  numeric  abs_spearman  0.6123    False
2          pts_l3  numeric  abs_spearman  0.6105    False
3   snap_share_l3  numeric  abs_spearman  0.5667    False
4          xfp_l8  numeric  abs_spearman  0.5599    False
5          xfp_l3  numeric  abs_spearman  0.5579    False
6      pts_max_l8  numeric  abs_spearman  0.5565    False
7  team_pos_share  numeric  abs_spearman  0.5547    False
8   snap_share_l8  numeric  abs_spearman  0.5483    False
9          fp_ecr  numeric  abs_spearman  0.5246    False


**Findings**


- 0 constant and 0 duplicate features. 2 pairs above 0.95: `fp_pos_rank`~`sleeper_pos_adp` (two yearly sources that mostly agree) and `pts_std_l8`~`pts_max_l8`. Both are kept for the tree model.
- **No leakage suspect:** the strongest single feature is `pts_season_avg` (|Spearman| 0.63).

## 7. Validation
Frame consistency, drift between train and test, and a quick baseline check.

In [9]:
print(dp.check_frames(X, cat_features=fp.cat_features()))
drift = dp.drift_report(X["train"], X["test"], exclude=fp.target_encoded_features())
drift.head(8)

   split        issue                                                                                     detail
0  train  NaN present  37 columns, e.g. ['years_exp', 'pts_l3', 'snap_share_l3', 'tgt_share_l3', 'rush_att_l3...
1    val  NaN present  36 columns, e.g. ['pts_l3', 'snap_share_l3', 'tgt_share_l3', 'rush_att_l3', 'pass_att_...
2   test  NaN present  34 columns, e.g. ['pts_l3', 'snap_share_l3', 'tgt_share_l3', 'rush_att_l3', 'pass_att_...


,feature,psi,flag
0,week,6.761797,major
1,games_season,4.519215,major
2,total_line,0.184009,moderate
3,qb_ppr_rel_l8,0.148790,moderate
4,opp_pass_epa_allowed_rel_l8,0.111720,moderate
5,opp_points_allowed_rel_l8,0.111334,moderate
6,team_plays_rel_l8,0.089220,ok
7,rest,0.073768,ok


In [10]:
base = dp.baseline_check(X["train"], y["train"], X["val"], y["val"], "regression")
print(base)
import numpy as np
bench = pd.read_parquet("../data/benchmark_sleeper.parquet")  # benchmark only, never a feature
for k in ("val", "test"):
    s = frames[k][["player_id", "season", "week", TARGET]].merge(bench, on=["player_id", "season", "week"], how="left")
    print(f"Sleeper on {k} (same rows): RMSE {np.sqrt(((s.sleeper_proj - s.act) ** 2).mean()):.3f}, "
          f"MAE {(s.sleeper_proj - s.act).abs().mean():.3f}")

     model       mae        r2
0    dummy  5.434962 -0.204543
1  hist_gb  3.626264  0.504685
Sleeper on val (same rows): RMSE 5.084, MAE 3.243
Sleeper on test (same rows): RMSE 5.137, MAE 3.318


**Findings**


- Frames are consistent, and NaN appears only in the history, ranking and injury columns.
- **Drift train vs test:** `week` (PSI 6.8) and `games_season` (4.5) are by design, because the test covers weeks 1-7 only, so players have played fewer games. `total_line` 0.18, `qb_ppr_rel_l8` 0.15 and the league-relative defense values (about 0.11) are moderate. Everything else is under 0.1.
- **Baseline check on 2025:** an untuned HistGradientBoosting gets MAE 3.63 and R2 0.50 (dummy 5.43). **Sleeper on the same rows: MAE 3.24, RMSE 5.08.** Without weekly projections the gap is large. Much of it is availability: Sleeper's final projection is 0 for 96% of the players who end up not playing.

## 8. Outputs

In [11]:
paths = dp.save_frames({f"X_{k}_gbm": v for k, v in X.items()} | {f"y_{k}": v.to_frame(TARGET) for k, v in y.items()}, OUT)
clean_paths = dp.save_frames({k: v.drop(columns=[TARGET]) for k, v in splits.items()}, OUT, prefix="clean_")
paths |= dp.save_frames({f"eval_{k}": splits[k][pp.EVAL_COLUMNS + [TARGET]] for k in splits}, OUT)
dp.save_object(fp, f"{MODELS}/feature_processor_gbm.joblib")
dp.save_object(tp, f"{MODELS}/target_processor.joblib")
outputs = {"dir": OUT,
           "X": {"gbm": {k: paths[f"X_{k}_gbm"] for k in X}},
           "y": {k: paths[f"y_{k}"] for k in y},
           "clean": clean_paths,
           "eval": {k: paths[f"eval_{k}"] for k in splits},
           "processors": {"gbm": f"{MODELS}/feature_processor_gbm.joblib"},
           "target_processor": f"{MODELS}/target_processor.joblib",
           "processing_module": "processing_player_games.py"}
dp.write_processing_record("processing_record.yml", decisions=dec, cleaning_log=clean_log, target=tp,
                           split={"how": "time", "time_col": "season", "group_col": "player_id",
                                  "train": "2022-2024", "val": "2025", "test": "2026 weeks 1-7"},
                           feature_processors={"gbm": fp}, outputs=outputs,
                           notes=["season kept out of the features (no trend; trees cannot extrapolate an unseen season)",
                                  "no weekly projection is a feature; Sleeper's is read only from benchmark_sleeper.parquet",
                                  "eval_* frames hold player_id/season/week/position/act for joining the benchmark"])
print(paths)

{'X_train_gbm': '../data/processed\\X_train_gbm.parquet', 'X_val_gbm': '../data/processed\\X_val_gbm.parquet', 'X_test_gbm': '../data/processed\\X_test_gbm.parquet', 'y_train': '../data/processed\\y_train.parquet', 'y_val': '../data/processed\\y_val.parquet', 'y_test': '../data/processed\\y_test.parquet', 'eval_train': '../data/processed\\eval_train.parquet', 'eval_val': '../data/processed\\eval_val.parquet', 'eval_test': '../data/processed\\eval_test.parquet'}


## Summary


- Outputs in `../data/processed/`: `X_{train,val,test}_gbm` (71 features), `y_*`, `clean_*`, and `eval_*` (`player_id, season, week, position, act`, used to join the benchmark). Processors are in `../models/`, and the record is `processing_record.yml`.
- No weekly projection anywhere in the features. Sleeper is read only from `benchmark_sleeper.parquet` for scoring.